## NLP Final
### Part 2: Topic Detection - BERTopic (Text)
### Aren Mizuno
### March 12, 2026

In [1]:
# Imports + install
!pip -q install bertopic sentence-transformers umap-learn hdbscan

import numpy as np
import pandas as pd
import torch
import re

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from google.colab import drive, files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [2]:
# Cuda
torch.cuda.is_available()

True

In [3]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
# Mount drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
# Load the cleaned dataset
data_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/news_final_project_cleaned.parquet"
df = pd.read_parquet(data_path, engine="pyarrow")
df.head(2)

,url,date,title,text
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...


In [7]:
# Check shape
df.shape

(136233, 4)

In [8]:
# Text cleaning helpers
TOPIC_JUNK_PHRASES = [
    "open menu", "menu", "search", "login", "log in", "sign up", "sign in",
    "subscribe", "privacy policy", "terms of service", "terms and conditions",
    "contact us", "contact support", "newsletter", "skip to content",
    "follow us", "read the rules", "mobile app", "read more", "click here",
    "all rights reserved", "advertisement", "sponsored content",
    "yahoo finance", "reuters", "associated press", "ap news",
]

TOPIC_JUNK_WORDS = {
    "home", "world", "business", "finance", "sports", "weather",
    "entertainment", "store", "blog", "forums", "shop", "mail", "more"
}


def clean_text(s):
    s = str(s).lower()

    # remove phrase-level boilerplate first
    for phrase in TOPIC_JUNK_PHRASES:
        s = re.sub(rf"\b{re.escape(phrase)}\b", " ", s)

    # remove urls
    s = re.sub(r"http\S+|www\.\S+", " ", s)

    # remove timestamps like 23:08:29 PKT or 12:45 PM
    s = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?\s*(?:am|pm|[A-Z]{2,5})?\b", " ", s)

    # remove dates like 2024-07-01 or 07/01/2024
    s = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", " ", s)
    s = re.sub(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", " ", s)

    # remove ticker fragments like [BFRG] or $NVDA
    s = re.sub(r"\[[A-Z0-9.\-]{1,10}\]", " ", s)
    s = re.sub(r"\$[A-Za-z]{1,6}\b", " ", s)

    # remove weird concatenated site-chrome tokens
    s = re.sub(r"\b[A-Za-z]+(?:[A-Z][a-z]+){2,}\b", " ", s)

    # remove separator junk
    s = re.sub(r"[_|•\-–—/]+", " ", s)

    # keep only letters/spaces
    s = re.sub(r"[^a-z\s]", " ", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    # remove single-word junk after normalization
    words = [w for w in s.split() if w not in TOPIC_JUNK_WORDS]

    # optional: drop 1-character tokens
    words = [w for w in words if len(w) > 1]

    return " ".join(words)

df["text"] = df["text"].map(clean_text)

In [9]:
# Set up embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# UMAP reduction
umap_model = UMAP(
    n_components=5,
    n_neighbors=100,
    min_dist=0.1,
    metric="cosine",
    random_state=0
)


In [11]:
# HDBSCAN clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=975,
    min_samples=98,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [12]:
# c-TF-IDF vectorizer
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=30_000    # prevents vocab explosion
)

In [13]:
# Topic representation
ctfidf_model = ClassTfidfTransformer()

In [14]:
# BERTopic model
bert_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    verbose=True
)

# Fit + transform
docs = df["text"].astype(str).tolist()
topics, probs = bert_model.fit_transform(docs)
df["bertopic_topic_text"] = topics

2026-03-11 21:25:17,303 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/4258 [00:00<?, ?it/s]

2026-03-11 21:32:50,649 - BERTopic - Embedding - Completed ✓
2026-03-11 21:32:50,654 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 21:41:47,522 - BERTopic - Dimensionality - Completed ✓
2026-03-11 21:41:47,526 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 21:42:02,888 - BERTopic - Cluster - Completed ✓
2026-03-11 21:42:02,924 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 21:46:45,340 - BERTopic - Representation - Completed ✓


In [15]:
# Number of topics
topic_info = bert_model.get_topic_info()
len(topic_info)

18

In [16]:
# Topic summary
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,36170,-1_ai_news_new_data,"[ai, news, new, data, share, said, technology,...",[build an ai strategy that survives first cont...
1,0,61119,0_ai_news_new_data,"[ai, news, new, data, technology, google, inte...",[white house unveils ai rules to address safet...
2,1,6398,1_ago_news_hours ago_hours,"[ago, news, hours ago, hours, video, stories, ...",[professor attempts to fail students after fal...
3,2,5560,2_share_ai_market_india,"[share, ai, market, india, stock, bank, new, n...",[openai latest breakthrough is sobering realit...
4,3,3572,3_newswires_presswire_ein presswire_ein,"[newswires, presswire, ein presswire, ein, rel...",[biontech to host ai day as an edition of its ...
5,4,3270,4_ment_products_services_overviewview,"[ment, products, services, overviewview, consu...",[new fintech startup economyx ai uses machine ...
6,5,3067,5_health_ai_healthcare_medical,"[health, ai, healthcare, medical, patients, pa...",[artificial intelligence in oncology market to...
7,6,2808,6_share price_price_nasdaq_share,"[share price, price, nasdaq, share, symbols, m...",[stock split artificial intelligence ai stocks...
8,7,2802,7_npr_radio_public_news,"[npr, radio, public, news, schedule, donate, m...",[bubbling questions about the limitations of a...
9,8,1951,8_gray_ai_gray media_prnewswire,"[gray, ai, gray media, prnewswire, media group...",[genpact integrates generative ai into enterpr...


In [17]:
# Save as parquet
save_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_info_text.parquet"
topic_info.to_parquet(save_path, engine="pyarrow", compression="snappy")
print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_info_text.parquet


In [18]:
df.head(5)

,url,date,title,text,bertopic_topic_text
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",bad idea ai price bad market cap price today c...,10
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,this ai video of gymnastics might be the freak...,0
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,"If using AI feels like a chore, try this - Boi...",if using ai feels like chore try this boing bo...,0
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,The Road Ahead: How China's AI Foundation Mode...,the road ahead how china ai foundation model i...,0
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,Microsoft and Nvidia to Empower Developers wit...,microsoft and nvidia to empower developers wit...,0


In [19]:
# Save as parquet
save_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_text.parquet"
df.to_parquet(save_path, engine="pyarrow", index=False)
print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_text.parquet
